# OSNAP

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import xarray as xr
import iconspy as ispy
from iconspy.tests.conftest import get_ds_tgrid_lr

## Grid setup
Load the ICON grid file and produce an ICONSPy dataset

In [ ]:
ds_tgrid = get_ds_tgrid_lr()  # This can be replaced with your own model tgrid
ds_IsD = ispy.convert_tgrid_data(ds_tgrid)
ds_IsD

## Get the coordinates of the OSNAP-Array

In [ ]:
# Open the mooring location files
import numpy as np
ds_osnap_west = xr.open_dataset(
    "https://swift.dkrz.de/v1/dkrz_7fa6baba-db43-4d12-a295-8e3ebb1a01ed/iconspy_test_data/OSNAP_mooring_positions_west.nc?temp_url_sig=0a2fb0675ab8e26fc5059903e5fb70b1a042c598&temp_url_expires=2036-07-11T11:41:31Z",
    engine="h5netcdf"
)

ds_osnap_east = xr.open_dataset(
    "https://swift.dkrz.de/v1/dkrz_7fa6baba-db43-4d12-a295-8e3ebb1a01ed/iconspy_test_data/OSNAP_mooring_positions_east.nc?temp_url_sig=e700534e9ce072aab794f0f6b6fd2a6b49f98f34&temp_url_expires=2036-07-11T11:42:22Z",
    engine="h5netcdf"
)

In [ ]:
# Visualise the mooring positions
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

ax.plot(
    ds_osnap_west["lon_sect"],
    ds_osnap_west["lat_sect"],
    transform=ccrs.PlateCarree(),
)

ax.plot(
    ds_osnap_east["lon_sect"],
    ds_osnap_east["lat_sect"],
    transform=ccrs.PlateCarree(),
)

ax.coastlines()

## OSNAP-West
### Find stations
We begin by finding the stations which make up OSNAP-West and discarding any stations which duplicate vertices

In [ ]:
# Create a list of the target stations at OSNAP West
_osnap_west_model_stations = []
for mooring in ds_osnap_west["mooring_name"].values:
    # Make the boundary stations as such
    if mooring.startswith("OSNAP_West"):
        boundary = True
    else:
        boundary = None
    
    _osnap_west_model_stations += [
        ispy.TargetStation(
            name=mooring,
            lon=float(ds_osnap_west["lon_sect"].sel(mooring_name=mooring).values),
            lat=float(ds_osnap_west["lat_sect"].sel(mooring_name=mooring).values),
            boundary=boundary,
        ).to_model_station(ds_IsD)
    ]
    


# Remove repeat stations
osnap_west_model_stations = []
osnap_west_model_stations_vertices = []
for model_station in _osnap_west_model_stations:
    if model_station.vertex not in osnap_west_model_stations_vertices:
        osnap_west_model_stations.append(model_station)
        osnap_west_model_stations_vertices.append(model_station.vertex)
        print(f"Added {model_station.name} at vertex {model_station.vertex}")
    else:
        print(f"Skipped {model_station.name} at vertex {model_station.vertex} because it is a duplicate")

# Visualise the model stations on a map
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

for i, model_station in enumerate(osnap_west_model_stations):
    ax.text(
        model_station.model_lon,
        model_station.model_lat,
        f"{i}",
        transform=ccrs.PlateCarree(),
        color="red",
    )
    
ax.coastlines()
ax.set_extent([-60, -40, 50, 70])

### Construct section

In [ ]:
# First we generate a list of sections connecting the model stations
osnap_west_sections = []
for i in range(len(osnap_west_model_stations) - 1):
    model_station_a = osnap_west_model_stations[i]
    model_station_b = osnap_west_model_stations[i + 1]
        
    name = str(model_station_a.name) + " to " + str(model_station_b.name)
    print(name)
    osnap_west_sections.append(
        ispy.Section(
            name=name,
            model_station_a=model_station_a,
            model_station_b=model_station_b,
            ds_IsD=ds_IsD,
            section_type="great circle",
        )
    )
    
osnap_west_sections

In [ ]:
# Now we combine these sections and visualise
osnap_west_combined = ispy.CombinedSection("OSNAP West", osnap_west_sections, ds_IsD)
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
osnap_west_combined.plot(ax=ax)
ax.set_extent([-60, -40, 50, 70])

## OSNAP-East

In [ ]:
# Create a list of the target stations at OSNAP East
_osnap_east_model_stations = []
for mooring in ds_osnap_east["mooring_name"].values:
    # Make the boundary stations as such
    if mooring.startswith("OSNAP_East"):
        boundary = True
    else:
        boundary = None
    
    _osnap_east_model_stations += [
        ispy.TargetStation(
            name=mooring,
            lon=float(ds_osnap_east["lon_sect"].sel(mooring_name=mooring).values),
            lat=float(ds_osnap_east["lat_sect"].sel(mooring_name=mooring).values),
            boundary=boundary,
        ).to_model_station(ds_IsD)
    ]

# Remove repeat stations
osnap_east_model_stations = []
osnap_east_model_stations_vertices = []
for model_station in _osnap_east_model_stations:
    # LS2 is one edge away from LS1 which causes things to break.
    # Need to fix ispy to cope with this
    if model_station.name.name == "LS2":
        print(f"Skipped {model_station.name} at vertex {model_station.vertex} because it is a near duplicate")
    elif model_station.vertex not in osnap_east_model_stations_vertices:
        osnap_east_model_stations.append(model_station)
        osnap_east_model_stations_vertices.append(model_station.vertex)
        print(f"Added {model_station.name} at vertex {model_station.vertex}")
    else:
        print(f"Skipped {model_station.name} at vertex {model_station.vertex} because it is a duplicate")

# Visualise the model stations on a map
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

for i, model_station in enumerate(osnap_east_model_stations):
    ax.text(
        model_station.model_lon,
        model_station.model_lat,
        f"{i}",
        transform=ccrs.PlateCarree(),
        color="red",
    )
    
ax.coastlines()
ax.set_extent([-60, 0, 50, 70])

In [ ]:
osnap_east_sections = []
for i in range(len(osnap_east_model_stations) - 1):
    model_station_a = osnap_east_model_stations[i]
    model_station_b = osnap_east_model_stations[i + 1]
        
    name = str(model_station_a.name) + " to " + str(model_station_b.name)
    print(name)
    osnap_east_sections.append(
        ispy.Section(
            name=name,
            model_station_a=model_station_a,
            model_station_b=model_station_b,
            ds_IsD=ds_IsD,
            section_type="great circle",
        )
    )
    
osnap_east_sections

In [ ]:
osnap_east_combined = ispy.CombinedSection("OSNAP East", osnap_east_sections, ds_IsD)
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
osnap_east_combined.plot(ax=ax)
ax.set_extent([-60, 0, 50, 70])

## Output in xarray formats

In [ ]:
ds_osnap_east_model = osnap_east_combined.to_ispy_section()
ds_osnap_east_model

In [ ]:
ds_osnap_west_model = osnap_west_combined.to_ispy_section()
ds_osnap_west_model